# Notebook 2: Exact Method (Professor's Formulas)
Computes H, H_L, H_R, G, K by **looping over every data point**.
These are the ground truth values. Slower but exact.

```
H   = Σ|xᵢ - O₄| / n              (MAD around median)
H_L = Σ|xᵢ - O₂| / (n/2)          (left half MAD, xᵢ ≤ O₄)
H_R = Σ|xᵢ - O₆| / (n/2)          (right half MAD, xᵢ > O₄)
K   = (H_L + H_R) / (2H)           (kurtosis)
G   = (mean - median) / H           (skewness)
```

In [ ]:
!pip install -q transformers torch safetensors pandas matplotlib seaborn

In [ ]:
import re, os, gc, warnings
import numpy as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
OUT = 'outputs_exact'
os.makedirs(OUT, exist_ok=True)
print('Ready.')

In [ ]:
# ==============================================
# EXACT FORMULAS — Professor's definitions
# Loop over every single data point
# ==============================================

def compute_octiles(w):
    return np.quantile(w, [1/8, 2/8, 3/8, 4/8, 5/8, 6/8, 7/8], method='linear')

def exact_metrics(w):
    """Compute all exact metrics by touching every data point."""
    w = np.asarray(w, dtype=np.float64).ravel()
    O = compute_octiles(w)
    O1,O2,O3,O4,O5,O6,O7 = O

    # H = mean absolute deviation from median
    H = np.mean(np.abs(w - O4))

    # Split into left and right halves
    left  = w[w <= O4]
    right = w[w > O4]

    # H_L = MAD of left half from its median (O2)
    H_L = np.mean(np.abs(left - O2)) if len(left) > 0 else np.nan

    # H_R = MAD of right half from its median (O6)
    H_R = np.mean(np.abs(right - O6)) if len(right) > 0 else np.nan

    # K = (H_L + H_R) / (2H)
    K = (H_L + H_R) / (2*H) if H > 0 else np.nan

    # G = (mean - median) / H
    G = (np.mean(w) - O4) / H if H > 0 else np.nan

    return {
        'O1':O1,'O2':O2,'O3':O3,'O4':O4,'O5':O5,'O6':O6,'O7':O7,
        'H_exact': H,
        'H_L_exact': H_L,
        'H_R_exact': H_R,
        'G_exact': G,
        'K_exact': K,
    }

print('Exact formulas defined.')

In [ ]:
# ==============================================
# CLASSIFIER (same as notebook 1)
# ==============================================

def classify_param(name, family):
    if family == 'gpt2':
        m = re.search(r'h\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.attn.c_attn.weight' in name:  return layer, 'attn_input'
        if '.attn.c_proj.weight' in name:  return layer, 'attn_output'
        if '.mlp.c_fc.weight' in name:     return layer, 'ffn_input'
        if '.mlp.c_proj.weight' in name:   return layer, 'ffn_output'
    elif family == 'opt':
        m = re.search(r'layers\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.self_attn.q_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.k_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.v_proj.weight' in name: return layer, 'attn_input'
        if '.self_attn.out_proj.weight' in name: return layer, 'attn_output'
        if '.fc1.weight' in name: return layer, 'ffn_input'
        if '.fc2.weight' in name: return layer, 'ffn_output'
    elif family == 'pythia':
        m = re.search(r'layers\.(\d+)\.', name)
        layer = int(m.group(1)) if m else -1
        if '.attention.query_key_value.weight' in name: return layer, 'attn_input'
        if '.attention.dense.weight' in name: return layer, 'attn_output'
        if '.mlp.dense_h_to_4h.weight' in name: return layer, 'ffn_input'
        if '.mlp.dense_4h_to_h.weight' in name: return layer, 'ffn_output'
    return -1, None

print('Classifier defined.')

In [ ]:
# ==============================================
# ANALYSE ONE MODEL (exact — slower)
# ==============================================

def analyse_model_exact(model_name, family, label):
    from transformers import AutoModelForCausalLM
    print(f'\nLoading {model_name} ...')
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()
    rows = []
    for name, p in model.named_parameters():
        layer, mtype = classify_param(name, family)
        if mtype is None: continue
        w = p.detach().cpu().numpy().astype(np.float64).ravel()
        m = exact_metrics(w)
        m['model'] = label
        m['param'] = name
        m['layer'] = layer
        m['type'] = mtype
        m['shape'] = str(tuple(p.shape))
        m['n'] = w.size
        rows.append(m)
    del model; gc.collect()
    df = pd.DataFrame(rows)
    print(f'  Done: {len(df)} matrices for {label}')
    return df

print('Exact analysis function defined.')

In [ ]:
# ==============================================
# RUN ALL MODELS
# ==============================================

MODELS = [
    ('openai-community/gpt2',        'gpt2',   'GPT2-Small'),
    ('openai-community/gpt2-medium', 'gpt2',   'GPT2-Medium'),
    ('facebook/opt-125m',            'opt',     'OPT-125M'),
    ('EleutherAI/pythia-160m',       'pythia',  'Pythia-160M'),
]

dfs = [analyse_model_exact(n, f, l) for n, f, l in MODELS]
df = pd.concat(dfs, ignore_index=True)
df.to_csv(f'{OUT}/all_exact.csv', index=False)
print(f'\nTotal: {len(df)} matrices. Saved all_exact.csv')

In [ ]:
# ==============================================
# TABLE: Summary (means by model × type)
# ==============================================

summary = df.groupby(['model','type'])[['H_exact','H_L_exact','H_R_exact','G_exact','K_exact']].mean().round(5).reset_index()
summary.to_csv(f'{OUT}/summary_exact.csv', index=False)
print('=== Exact Metrics ===')
display(summary)

In [ ]:
# ==============================================
# FIG 1: Histograms — GPT2-Small attn_output layers 0,4,8,11
# ==============================================

from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained('openai-community/gpt2')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('GPT2-Small attn_output: exact metrics at different depths', fontsize=13)

for ax, lyr in zip(axes.flat, [0, 4, 8, 11]):
    pname = f'transformer.h.{lyr}.attn.c_proj.weight'
    w = dict(model.named_parameters())[pname].detach().cpu().numpy().ravel()
    lo, hi = np.percentile(w, [0.5, 99.5])
    ax.hist(w, bins=200, range=(lo, hi), density=True, color='steelblue', alpha=0.7, edgecolor='none')
    O = compute_octiles(w)
    for oi in O: ax.axvline(oi, color='coral', lw=0.8, alpha=0.6)
    ax.axvline(O[3], color='red', lw=1.5)
    m = exact_metrics(w)
    ax.set_title(f'Layer {lyr}:  H={m["H_exact"]:.4f}  G={m["G_exact"]:.4f}  K={m["K_exact"]:.3f}', fontsize=10)
    ax.set_xlabel('Weight value')

del model; gc.collect()
plt.tight_layout()
plt.savefig(f'{OUT}/fig1_histograms_exact.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 2: H_exact vs layer depth (all 4 models)
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ntype in zip(axes, ['attn_output', 'ffn_output']):
    sub = df[df['type'] == ntype]
    for label in sub['model'].unique():
        s = sub[sub['model']==label].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['H_exact'], marker='o', ms=4, label=label)
    ax.set_title(f'{ntype}: H (Exact) vs depth')
    ax.set_xlabel('Normalised layer depth'); ax.set_ylabel('H_exact')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig2_H_exact_vs_depth.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 3: K_exact vs layer depth
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ntype in zip(axes, ['attn_output', 'ffn_output']):
    sub = df[df['type'] == ntype]
    for label in sub['model'].unique():
        s = sub[sub['model']==label].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['K_exact'], marker='o', ms=4, label=label)
    ax.set_title(f'{ntype}: K (Exact) vs depth')
    ax.set_xlabel('Normalised layer depth'); ax.set_ylabel('K_exact')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig3_K_exact_vs_depth.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 4: G_exact vs layer depth
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ntype in zip(axes, ['attn_output', 'ffn_output']):
    sub = df[df['type'] == ntype]
    for label in sub['model'].unique():
        s = sub[sub['model']==label].sort_values('layer')
        mx = s['layer'].max()
        ax.plot(s['layer']/max(mx,1), s['G_exact'], marker='o', ms=4, label=label)
    ax.axhline(0, color='gray', ls='--', alpha=0.4)
    ax.set_title(f'{ntype}: G (Exact) vs depth')
    ax.set_xlabel('Normalised layer depth'); ax.set_ylabel('G_exact')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig4_G_exact_vs_depth.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 5: Heatmaps of H_exact (2x2)
# ==============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('H (Exact MAD) by layer and matrix type', fontsize=13)
for ax, (_, _, label) in zip(axes.flat, MODELS):
    sub = df[df['model']==label]
    pv = sub.pivot_table(index='type', columns='layer', values='H_exact', aggfunc='mean')
    sns.heatmap(pv, cmap='viridis', ax=ax, cbar_kws={'label':'H_exact'})
    ax.set_title(label); ax.set_xlabel('Layer'); ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{OUT}/fig5_heatmap_H_exact.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 6: Heatmaps of K_exact (2x2)
# ==============================================

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('K (Exact Kurtosis) by layer and matrix type', fontsize=13)
for ax, (_, _, label) in zip(axes.flat, MODELS):
    sub = df[df['model']==label]
    pv = sub.pivot_table(index='type', columns='layer', values='K_exact', aggfunc='mean')
    sns.heatmap(pv, cmap='magma', ax=ax, cbar_kws={'label':'K_exact'})
    ax.set_title(label); ax.set_xlabel('Layer'); ax.set_ylabel('')
plt.tight_layout()
plt.savefig(f'{OUT}/fig6_heatmap_K_exact.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 7: Boxplots
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='type', y='K_exact', hue='model', ax=axes[0])
axes[0].set_title('K (Exact) by matrix type'); axes[0].set_xlabel('')
sns.boxplot(data=df, x='type', y='G_exact', hue='model', ax=axes[1])
axes[1].set_title('G (Exact) by matrix type'); axes[1].set_xlabel('')
axes[1].axhline(0, color='gray', ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig(f'{OUT}/fig7_boxplots_exact.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# FIG 8: H_L and H_R across layers (unique to exact method)
# ==============================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, ntype in zip(axes, ['attn_output', 'ffn_output']):
    sub = df[(df['type']==ntype) & (df['model']=='GPT2-Small')].sort_values('layer')
    ax.plot(sub['layer'], sub['H_L_exact'], 'o-', label='H_L (left tail spread)', color='blue')
    ax.plot(sub['layer'], sub['H_R_exact'], 's-', label='H_R (right tail spread)', color='red')
    ax.plot(sub['layer'], sub['H_exact'], '^-', label='H (total spread)', color='green')
    ax.set_title(f'GPT2-Small {ntype}: H, H_L, H_R')
    ax.set_xlabel('Layer'); ax.set_ylabel('Value')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT}/fig8_HL_HR_depth.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================
# PRINT SUMMARY
# ==============================================

print('='*60)
print('KEY FINDINGS (Exact Method)')
print('='*60)
for label in df['model'].unique():
    sub = df[df['model']==label]
    print(f'\n--- {label} ---')
    for nt in ['attn_output','ffn_output']:
        s = sub[sub['type']==nt].sort_values('layer')
        if len(s)>0:
            print(f'  {nt}: H [{s.H_exact.min():.4f} → {s.H_exact.max():.4f}]  K [{s.K_exact.min():.3f} → {s.K_exact.max():.3f}]')
    print(f'  K range overall: [{sub.K_exact.min():.3f}, {sub.K_exact.max():.3f}]  (should be 0-1)')

In [ ]:
try:
    from google.colab import files
    import glob
    for f in glob.glob(f'{OUT}/*'): files.download(f)
except: print('Files in:', OUT, os.listdir(OUT))